In [13]:
import numpy as np
import data.breathe_data as bd
import models.helpers as mh
import model_validation.model_evidence as me
from scipy.stats import wilcoxon, chi2

In [ ]:
df = bd.load_meas_from_excel("TR5_O2_FEV1_FEF2575_with_idx_28092025", study_folder="TR")
df["idx ecFEF2575%ecFEV1"] = df["idx ecFEF25-75 % ecFEV1 (%)"]

INFO:root:* Checking for same day measurements *


# Model evidence

In [ ]:
s1 = df.groupby("ID").apply(
    lambda dftmp: me.run_ve(dftmp.head(1), with_fef2575=False, with_rmax=False)
)

In [11]:
s3 = df.groupby("ID").apply(
    lambda dftmp: me.run_ve(dftmp.head(1), with_fef2575=True, with_rmax=False)
)

In [14]:
# Wilcoxon positive-rank test for paired samples
# Paired comparison of log-likelihoods
stat_13, pval_13 = wilcoxon(np.log(s3), np.log(s1), alternative="greater")

m13 = (s3 - s1) / s1 * 100
tot = len(s1)


def compare(full, nested, m, pval, tot):
    print(
        f"{full} - {nested} median = +{m.median():.4f}% ({pval:.0e}), >0%: {(m>0).sum()/tot*100:.0f}% ({(m>0).sum()}), >30%: {(m>30).sum()/tot*100:.0f}% ({(m>30).sum()}), >100%: {(m>100).sum()/tot*100:.0f}% ({(m>100).sum()})"
    )


compare("P(FEV1|M, FEF2575)", "baseline", m13, pval_13, tot)

P(FEV1|M, FEF2575) - baseline median = +34.9955% (4e-96), >0%: 64% (3257), >30%: 52% (2640), >100%: 29% (1471)


In [15]:
import numpy as np
import scipy.stats as st

# x, y are numpy arrays of your log-transformed observations
# n ~ 352 each
x = np.log(s1)
y = np.log(s3)

# 1) Welch t-test
tstat, pval = st.ttest_ind(x, y, equal_var=False)
stat_12, pval_12 = wilcoxon(y, x, alternative="greater")
print("Welch t:", tstat, "p =", pval)
print("Wilcoxon:", stat_12, "p =", pval_12)


# 2) Bootstrap 95% CI for difference in means (10000 resamples)
def bootstrap_diff_means(x, y, n_boot=10000):
    rng = np.random.default_rng()
    diffs = np.empty(n_boot)
    nx, ny = len(x), len(y)
    for i in range(n_boot):
        bx = rng.choice(x, size=nx, replace=True)
        by = rng.choice(y, size=ny, replace=True)
        diffs[i] = bx.mean() - by.mean()
    return np.percentile(diffs, [2.5, 97.5]), diffs


ci, diffs = bootstrap_diff_means(x, y)
print("Bootstrap 95% CI for mean difference (log-scale):", ci)


# 3) Permutation test for difference in means (5000 perms)
def perm_test_mean(x, y, n_perm=5000):
    rng = np.random.default_rng()
    obs = x.mean() - y.mean()
    pooled = np.concatenate([x, y])
    count = 0
    n = len(x)
    for _ in range(n_perm):
        rng.shuffle(pooled)
        if pooled[:n].mean() - pooled[n:].mean() >= abs(obs):  # two-sided
            count += 1
    p_emp = (count + 1) / (n_perm + 1)
    return obs, p_emp


obs_diff, p_perm = perm_test_mean(x, y)
print("Observed diff (log-scale):", obs_diff, "Permutation p (approx):", p_perm)

# 4) Convert log-difference to ratio (original scale)
log_diff = x.mean() - y.mean()
ratio = np.exp(log_diff)
print("Estimated geometric mean ratio (group x / group y):", ratio)

Welch t: -7.476028680614389 p = 8.503849879491382e-14
Wilcoxon: 8703596.0 p = 4.215117986167297e-96
Bootstrap 95% CI for mean difference (log-scale): [-0.21294819 -0.12427227]
Observed diff (log-scale): -0.1700970292543511 Permutation p (approx): 0.0001999600079984003
Estimated geometric mean ratio (group x / group y): 0.8435829603995941


# AR inference

In [4]:
df_res = bd.load_meas_from_excel(
    "infer_AR_using_fev1_fef2575_TR5_25092025",
    ["Airway resistance (%)", "Healthy FEV1 (L)"],
    bypass_sanity_checks=True,
    study_folder="TR",
)

In [5]:
# cols2keep = [
#     "ID",
#     "Age",
#     "Sex",
#     "Height",
#     "FEV1",
#     "FEF2575",
#     "FEV1 % Predicted",
# ]
# df_res = df_res.merge(df[cols2keep], how="inner", on="ID")
df_res = df_res.merge(df, how="inner", on="ID")

In [ ]:
AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, prior={"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df_res[AC.name] = df_res[AR.name].apply(lambda row: row[::-1])

df_res["AC std"] = df_res[AC.name].apply(lambda row: AC.get_std(row))
df_res["AC mean"] = df_res[AC.name].apply(lambda row: AC.get_mean(row))
df_res["AC low"] = df_res["AC mean"] - df_res["AC std"]
df_res["AC high"] = df_res["AC mean"] + df_res["AC std"]

In [ ]:
low_ac = df_res["FEV1 % Predicted"] < df_res["AC low"]
high_ac = df_res["FEV1 % Predicted"] > df_res["AC high"]
agree = (df_res["FEV1 % Predicted"] >= df_res["AC low"]) & (
    df_res["FEV1 % Predicted"] <= df_res["AC high"]
)

print(f"{low_ac.sum()} low airway conductance")
print(f"{high_ac.sum()} high airway conductance")
print(f"{agree.sum()} agree")
print(low_ac.sum() + high_ac.sum() + agree.sum())

69 low airway conductance
1895 high airway conductance
3141 agree
5105


In [33]:
df[low_ac]

,ID,UID,Age,Sex,Height,Health,Asthma,Bronchitis,Smoke daily,Cigarettes number,...,FEF2575,Predicted FEV1,Healthy O2 Saturation,FEV1 % Predicted,O2 Saturation % Healthy,FEF2575%FEV1,Date Recorded,idx ecFEV1 (L),idx ecFEF25-75 % ecFEV1 (%),idx O2 saturation (%)
68,80,176,67,Male,173.4,2.0,0.0,0.0,2.0,7,...,1.997,3.145943,97.142930,47.108286,96.074928,134.750337,2001-01-01,29,67,43
569,668,1374,69,Male,163.0,2.0,0.0,0.0,2.0,NaN,...,1.370,2.681631,97.329474,39.528187,98.634048,129.245283,2001-01-01,21,64,46
765,911,1853,70,Male,168.1,2.0,0.0,0.0,3.0,NaN,...,3.176,2.839285,97.237996,74.737136,100.783649,149.670123,2001-01-01,42,74,48
959,1135,2291,65,Male,168.8,2.0,0.0,0.0,2.0,NaN,...,3.443,3.030168,97.225440,80.127569,98.739589,141.803954,2001-01-01,48,70,46
1313,1577,3209,65,Male,177.0,2.0,0.0,1.0,1.0,10,...,3.381,3.366606,97.078356,73.842913,96.828998,136.001609,2001-01-01,49,68,44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4859,7097,19985,80,Female,171.3,2.0,0.0,0.0,3.0,NaN,...,2.520,2.159882,98.001549,73.568850,98.641298,158.590308,2001-01-01,31,79,47
4888,7140,20128,80,Female,158.6,2.0,0.0,0.0,3.0,NaN,...,2.000,1.834300,98.229349,79.212796,100.102466,137.646249,2001-01-01,29,68,48
4976,7270,20447,73,Female,165.9,2.0,0.0,0.0,2.0,3,...,1.522,2.216135,98.098409,45.800468,96.158542,149.950739,2001-01-01,20,74,44
5011,7320,20609,78,Female,158.7,3.0,0.0,0.0,2.0,20,...,1.144,1.886083,98.227555,56.042071,97.050161,108.230842,2001-01-01,21,54,45
